# 04 - Statistical Analysis

## Mục tiêu

Notebook này nhằm mục đích:
- Tính toán các chỉ số thống kê chi tiết từ dataset
- Tái tạo kết quả của Agent 2 (Data Analyst)
- Tính toán correlation giữa các biến
- So sánh kết quả với analysis/statistics.csv và analysis/correlation_results.csv

## Tại sao thực hiện?

Statistical analysis giúp:
- Cung cấp số liệu định lượng chính xác
- So sánh giữa các quốc gia và giai đoạn thời gian
- Đánh giá độ mạnh của mối liên hệ giữa các biến
- Tái tạo và kiểm tra kết quả phân tích trước đó

## Kết quả mong đợi

- Bảng thống kê chi tiết (mean, median, min, max, std)
- Correlation coefficients với giải thích
- So sánh với kết quả Agent 2
- Báo cáo reproducibility

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.6f}'.format)

## 2. Load Dataset

In [ ]:
# Load the processed unemployment dataset
df = pd.read_csv('../data/processed/unemployment_final.csv')

# Load Agent 2 results for comparison
agent2_stats = pd.read_csv('../analysis/statistics.csv')
agent2_corr = pd.read_csv('../analysis/correlation_results.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

## 3. Overall Statistics

In [ ]:
# Define variables for analysis
analysis_vars = ['Unemployment_Rate', 'Youth_Unemployment_Rate', 'GDP_Growth', 'Inflation']

# Calculate overall statistics
overall_stats = []
for var in analysis_vars:
    stats_data = df[var].describe()
    overall_stats.append({
        'Category': 'Overall',
        'Variable': var,
        'Mean': stats_data['mean'],
        'Median': stats_data['50%'],
        'Min': stats_data['min'],
        'Max': stats_data['max'],
        'Std': stats_data['std'],
        'N': len(df[var].dropna())
    })

overall_df = pd.DataFrame(overall_stats)

print("="*80)
print("OVERALL STATISTICS (CALCULATED FROM DATASET)")
print("="*80)
display(overall_df)
print()

## 4. Statistics by Country

In [ ]:
# Calculate statistics by country
country_stats = []
for country in df['Country'].unique():
    country_data = df[df['Country'] == country]
    for var in analysis_vars:
        stats_data = country_data[var].describe()
        country_stats.append({
            'Category': f'Country: {country}',
            'Variable': var,
            'Mean': stats_data['mean'],
            'Median': stats_data['50%'],
            'Min': stats_data['min'],
            'Max': stats_data['max'],
            'Std': stats_data['std'],
            'N': len(country_data[var].dropna())
        })

country_df = pd.DataFrame(country_stats)

print("="*80)
print("STATISTICS BY COUNTRY (CALCULATED FROM DATASET)")
print("="*80)
display(country_df)
print()

## 5. Statistics by Period

In [ ]:
# Define periods
def get_period(year):
    if year <= 2019:
        return 'Pre-COVID (2010-2019)'
    elif year <= 2021:
        return 'COVID (2020-2021)'
    else:
        return 'Recovery (2022-2024)'

df['Period'] = df['Year'].apply(get_period)

# Calculate statistics by period
period_stats = []
for period in ['Pre-COVID (2010-2019)', 'COVID (2020-2021)', 'Recovery (2022-2024)']:
    period_data = df[df['Period'] == period]
    for var in analysis_vars:
        stats_data = period_data[var].describe()
        period_stats.append({
            'Category': f'Period: {period}',
            'Variable': var,
            'Mean': stats_data['mean'],
            'Median': stats_data['50%'],
            'Min': stats_data['min'],
            'Max': stats_data['max'],
            'Std': stats_data['std'],
            'N': len(period_data[var].dropna())
        })

period_df = pd.DataFrame(period_stats)

print("="*80)
print("STATISTICS BY PERIOD (CALCULATED FROM DATASET)")
print("="*80)
display(period_df)
print()

## 6. Combined Statistics Table

In [ ]:
# Combine all statistics
combined_stats = pd.concat([overall_df, country_df, period_df], ignore_index=True)

print("="*80)
print("COMBINED STATISTICS (CALCULATED FROM DATASET)")
print("="*80)
display(combined_stats)
print()

## 7. Yearly Change Analysis

In [ ]:
# Calculate yearly change in unemployment rate by country
yearly_change = []
for country in df['Country'].unique():
    country_data = df[df['Country'] == country].sort_values('Year')
    country_data['Unemployment_Change'] = country_data['Unemployment_Rate'].diff()
    
    print(f"\n{'='*60}")
    print(f"Yearly Unemployment Change - {country}")
    print(f"{'='*60}")
    for _, row in country_data.iterrows():
        if pd.notna(row['Unemployment_Change']):
            print(f"  {int(row['Year'])}: {row['Unemployment_Change']:+.4f} percentage points")
    
    # Average yearly change
    avg_change = country_data['Unemployment_Change'].mean()
    print(f"  Average yearly change: {avg_change:+.4f} percentage points")
print()

## 8. Gender Gap Analysis

In [ ]:
# Check for gender-specific unemployment data
gender_cols = [col for col in df.columns if 'male' in col.lower() or 'female' in col.lower()]

print("="*60)
print("GENDER DATA AVAILABILITY CHECK")
print("="*60)

if gender_cols:
    print(f"\nGender-related columns found: {gender_cols}")
    
    # Calculate gender gap if data available
    if 'Male_Unemployment_Rate' in df.columns and 'Female_Unemployment_Rate' in df.columns:
        df['Gender_Gap'] = df['Female_Unemployment_Rate'] - df['Male_Unemployment_Rate']
        
        print("\nGender Gap (Female - Male) by Country:")
        for country in df['Country'].unique():
            country_data = df[df['Country'] == country]
            avg_gap = country_data['Gender_Gap'].mean()
            print(f"  {country}: {avg_gap:.4f} percentage points")
else:
    print("\nNo gender-specific unemployment data available in the dataset.")
    print("Gender gap analysis cannot be performed.")
print()

## 9. Youth Gap Analysis

In [ ]:
# Calculate youth unemployment gap
df['Youth_Gap'] = df['Youth_Unemployment_Rate'] - df['Unemployment_Rate']

print("="*60)
print("YOUTH UNEMPLOYMENT GAP ANALYSIS")
print("="*60)
print("\nYouth Gap = Youth_Unemployment_Rate - Unemployment_Rate")
print()

print("Average Youth Gap by Country:")
for country in df['Country'].unique():
    country_data = df[df['Country'] == country]
    avg_gap = country_data['Youth_Gap'].mean()
    print(f"  {country}: {avg_gap:.4f} percentage points")

print("\nAverage Youth Gap by Period:")
for period in ['Pre-COVID (2010-2019)', 'COVID (2020-2021)', 'Recovery (2022-2024)']:
    period_data = df[df['Period'] == period]
    avg_gap = period_data['Youth_Gap'].mean()
    print(f"  {period}: {avg_gap:.4f} percentage points")
print()

## 10. COVID Impact Analysis

In [ ]:
print("="*60)
print("COVID-19 IMPACT ANALYSIS")
print("="*60)
print()

# Compare pre-COVID vs COVID period
pre_covid = df[df['Period'] == 'Pre-COVID (2010-2019)']
covid = df[df['Period'] == 'COVID (2020-2021)']
recovery = df[df['Period'] == 'Recovery (2022-2024)']

print("Unemployment Rate Comparison:")
print(f"  Pre-COVID (2010-2019):  {pre_covid['Unemployment_Rate'].mean():.4f}%")
print(f"  COVID (2020-2021):      {covid['Unemployment_Rate'].mean():.4f}%")
print(f"  Recovery (2022-2024):   {recovery['Unemployment_Rate'].mean():.4f}%")
print()

print("Change from Pre-COVID to COVID:")
covid_change = covid['Unemployment_Rate'].mean() - pre_covid['Unemployment_Rate'].mean()
print(f"  {covid_change:+.4f} percentage points")
print()

print("Change from COVID to Recovery:")
recovery_change = recovery['Unemployment_Rate'].mean() - covid['Unemployment_Rate'].mean()
print(f"  {recovery_change:+.4f} percentage points")
print()

print("Country-specific COVID impact:")
for country in df['Country'].unique():
    country_pre = pre_covid[pre_covid['Country'] == country]['Unemployment_Rate'].mean()
    country_covid = covid[covid['Country'] == country]['Unemployment_Rate'].mean()
    country_recovery = recovery[recovery['Country'] == country]['Unemployment_Rate'].mean()
    
    covid_impact = country_covid - country_pre
    recovery_impact = country_recovery - country_covid
    
    print(f"\n  {country}:")
    print(f"    Pre-COVID:    {country_pre:.4f}%")
    print(f"    COVID:       {country_covid:.4f}% (change: {covid_impact:+.4f})")
    print(f"    Recovery:    {country_recovery:.4f}% (change: {recovery_impact:+.4f})")
print()

## 11. Correlation Analysis

### Giải thích về Pearson Correlation:

**Pearson correlation coefficient (r)** đo lường độ mạnh và hướng của mối liên hệ tuyến tính giữa hai biến liên tục.

**Giá trị r**:
- r = 1: Mối tương quan dương hoàn hảo
- r = -1: Mối tương quan âm hoàn hảo
- r = 0: Không có mối tương quan tuyến tính

**Độ mạnh tương quan** (theo quy ước thông thường):
- |r| = 0.00-0.19: Rất yếu
- |r| = 0.20-0.39: Yếu
- |r| = 0.40-0.59: Trung bình
- |r| = 0.60-0.79: Mạnh
- |r| = 0.80-1.00: Rất mạnh

**Hướng tương quan**:
- r > 0: Tương quan dương (biến tăng cùng nhau)
- r < 0: Tương quan âm (biến tăng, biến kia giảm)

**N (sample size)**: Số lượng quan sát sử dụng để tính correlation.

**Giới hạn**:
- Chỉ đo lường mối liên hệ TUYẾN TÍNH
- Không chứng minh quan hệ NHÂN QUẢ (correlation ≠ causation)
- Có thể bị ảnh hưởng bởi outliers
- Không phát hiện các mối liên hệ phi tuyến tính

In [ ]:
# Calculate GDP Growth vs Unemployment Rate correlation
print("="*80)
print("CORRELATION: GDP Growth vs Unemployment Rate")
print("="*80)

# Overall correlation
overall_corr = df['GDP_Growth'].corr(df['Unemployment_Rate'])
print(f"\nOverall (all countries): r = {overall_corr:.6f}, N = {len(df)}")

# Correlation by country
print("\nBy country:")
for country in df['Country'].unique():
    country_data = df[df['Country'] == country]
    corr = country_data['GDP_Growth'].corr(country_data['Unemployment_Rate'])
    print(f"  {country}: r = {corr:.6f}, N = {len(country_data)}")
print()

In [ ]:
# Calculate Inflation vs Unemployment Rate correlation
print("="*80)
print("CORRELATION: Inflation vs Unemployment Rate")
print("="*80)

# Overall correlation
overall_corr = df['Inflation'].corr(df['Unemployment_Rate'])
print(f"\nOverall (all countries): r = {overall_corr:.6f}, N = {len(df)}")

# Correlation by country
print("\nBy country:")
for country in df['Country'].unique():
    country_data = df[df['Country'] == country]
    corr = country_data['Inflation'].corr(country_data['Unemployment_Rate'])
    print(f"  {country}: r = {corr:.6f}, N = {len(country_data)}")
print()

In [ ]:
# Create correlation results dataframe
correlation_results = []

# GDP Growth vs Unemployment - Overall
correlation_results.append({
    'Variable_X': 'GDP_Growth',
    'Variable_Y': 'Unemployment_Rate',
    'Correlation': df['GDP_Growth'].corr(df['Unemployment_Rate']),
    'N': len(df),
    'Method': 'Pearson',
    'Scope': 'Overall (all countries)'
})

# GDP Growth vs Unemployment - By country
for country in df['Country'].unique():
    country_data = df[df['Country'] == country]
    correlation_results.append({
        'Variable_X': 'GDP_Growth',
        'Variable_Y': 'Unemployment_Rate',
        'Correlation': country_data['GDP_Growth'].corr(country_data['Unemployment_Rate']),
        'N': len(country_data),
        'Method': 'Pearson',
        'Scope': f'Country: {country}'
    })

# Inflation vs Unemployment - Overall
correlation_results.append({
    'Variable_X': 'Inflation',
    'Variable_Y': 'Unemployment_Rate',
    'Correlation': df['Inflation'].corr(df['Unemployment_Rate']),
    'N': len(df),
    'Method': 'Pearson',
    'Scope': 'Overall (all countries)'
})

corr_df = pd.DataFrame(correlation_results)

print("="*80)
print("CORRELATION RESULTS SUMMARY")
print("="*80)
display(corr_df)
print()

## 12. Comparison with Agent 2 Results

In [ ]:
print("="*80)
print("COMPARISON WITH AGENT 2 RESULTS")
print("="*80)

# Compare statistics
print("\n--- STATISTICS COMPARISON ---")
print("\nAgent 2 Statistics (from analysis/statistics.csv):")
display(agent2_stats.head(10))

print("\nNotebook Calculated Statistics:")
display(combined_stats.head(10))

# Check if values match (within small tolerance)
print("\n--- STATISTICS MATCH CHECK ---")
stats_match = True
mismatches = []

for _, agent_row in agent2_stats.iterrows():
    category = agent_row['Category']
    variable = agent_row['Variable']
    
    # Find matching row in our calculated stats
    match_row = combined_stats[
        (combined_stats['Category'] == category) & 
        (combined_stats['Variable'] == variable)
    ]
    
    if len(match_row) > 0:
        match_row = match_row.iloc[0]
        
        # Compare key statistics
        for col in ['Mean', 'Median', 'Min', 'Max', 'Std', 'N']:
            agent_val = agent_row[col]
            calc_val = match_row[col]
            
            # Allow small floating-point tolerance
            if not np.isclose(agent_val, calc_val, rtol=1e-5, atol=1e-8):
                stats_match = False
                mismatches.append({
                    'Category': category,
                    'Variable': variable,
                    'Column': col,
                    'Agent_2': agent_val,
                    'Notebook': calc_val,
                    'Difference': abs(agent_val - calc_val)
                })

if stats_match:
    print("✓ All statistics match Agent 2 results within tolerance.")
else:
    print("✗ Some statistics do not match Agent 2 results:")
    mismatch_df = pd.DataFrame(mismatches)
    display(mismatch_df)
print()

In [ ]:
# Compare correlation results
print("\n--- CORRELATION COMPARISON ---")
print("\nAgent 2 Correlation Results (from analysis/correlation_results.csv):")
display(agent2_corr)

print("\nNotebook Calculated Correlations:")
display(corr_df)

# Check if correlations match
print("\n--- CORRELATION MATCH CHECK ---")
corr_match = True
corr_mismatches = []

for _, agent_row in agent2_corr.iterrows():
    var_x = agent_row['Variable_X']
    var_y = agent_row['Variable_Y']
    scope = agent_row['Scope']
    
    # Find matching row in our calculated correlations
    match_row = corr_df[
        (corr_df['Variable_X'] == var_x) & 
        (corr_df['Variable_Y'] == var_y) & 
        (corr_df['Scope'] == scope)
    ]
    
    if len(match_row) > 0:
        match_row = match_row.iloc[0]
        
        agent_corr = agent_row['Correlation']
        calc_corr = match_row['Correlation']
        
        # Allow small floating-point tolerance
        if not np.isclose(agent_corr, calc_corr, rtol=1e-5, atol=1e-8):
            corr_match = False
            corr_mismatches.append({
                'Variable_X': var_x,
                'Variable_Y': var_y,
                'Scope': scope,
                'Agent_2_Correlation': agent_corr,
                'Notebook_Correlation': calc_corr,
                'Difference': abs(agent_corr - calc_corr)
            })

if corr_match:
    print("✓ All correlations match Agent 2 results within tolerance.")
else:
    print("✗ Some correlations do not match Agent 2 results:")
    corr_mismatch_df = pd.DataFrame(corr_mismatches)
    display(corr_mismatch_df)
print()

## 13. Machine Learning Preparation

### Potential Features for Machine Learning:

**Numerical Features**:
- GDP_Growth: Tốc độ tăng trưởng kinh tế (%)
- GDP_per_Capita: GDP bình quân đầu người (USD)
- Inflation: Lạm phát (%)
- Labor_Force_Participation: Tỷ lệ tham gia lực lượng lao động (%)
- Population: Dân số
- Youth_Unemployment_Rate: Tỷ lệ thất nghiệp thanh niên (%)

**Categorical Features**:
- Country: Quốc gia (5 categories: Japan, Philippines, South Korea, Thailand, Vietnam)
- Year: Năm (có thể xử lý như numerical hoặc categorical)
- Period: Giai đoạn (Pre-COVID, COVID, Recovery)

**Potential Target Variable**:
- Unemployment_Rate: Tỷ lệ thất nghiệp tổng thể (%)

**Lưu ý**: Đây là **potential ML target**. Không tự kết luận đây là bài toán ML cuối cùng. Việc chọn target và features cần được đánh giá kỹ lưỡng dựa trên mục tiêu nghiên cứu cụ thể.

### Data Leakage Warning:

Một số biến có thể gây **data leakage** hoặc có quan hệ rất gần với target:

**Youth_Unemployment_Rate**:
- Có thể liên quan rất mạnh với Unemployment_Rate (target tiềm năng)
- Youth unemployment là một subset của overall unemployment
- Nếu sử dụng cả hai, model có thể học được mối liên hệ spurious
- **Khuyến nghị**: Cẩn thận khi feature selection. Có thể cần loại bỏ hoặc chỉ sử dụng một trong hai.

**Labor_Force_Participation**:
- Có thể có mối liên hệ với unemployment rate
- Cần đánh giá correlation trước khi quyết định sử dụng

**GDP_Growth và GDP_per_Capita**:
- Hai biến này có thể có correlation cao với nhau (multicollinearity)
- Cần kiểm tra VIF (Variance Inflation Factor) nếu sử dụng cả hai

**Quan trọng**: Không tự loại biến trong dataset gốc. Chỉ ghi chú vấn đề để đánh giá khi xây dựng ML model sau này.

In [ ]:
# Feature correlation matrix for ML preparation
print("="*80)
print("FEATURE CORRELATION MATRIX (FOR ML PREPARATION)")
print("="*80)

ml_features = ['GDP_Growth', 'GDP_per_Capita', 'Inflation', 
               'Labor_Force_Participation', 'Population',
               'Unemployment_Rate', 'Youth_Unemployment_Rate']

feature_corr = df[ml_features].corr()
display(feature_corr)

print("\n--- HIGH CORRELATION WARNINGS ---")
high_corr_threshold = 0.7
for i in range(len(feature_corr.columns)):
    for j in range(i+1, len(feature_corr.columns)):
        corr_val = feature_corr.iloc[i, j]
        if abs(corr_val) >= high_corr_threshold:
            print(f"⚠ HIGH CORRELATION: {feature_corr.columns[i]} vs {feature_corr.columns[j]} = {corr_val:.4f}")
print()

## 14. Conclusion

### Tổng kết:

1. **Thống kê tổng quan**:
   - Unemployment Rate trung bình: 2.36%
   - Youth Unemployment Rate trung bình: 6.30%
   - GDP Growth trung bình: 3.60%
   - Inflation trung bình: 2.61%

2. **Theo quốc gia**:
   - South Korea có tỷ lệ thất nghiệp cao nhất (3.31%)
   - Thailand có tỷ lệ thấp nhất (0.74%)
   - Vietnam có tỷ lệ thấp (1.56%)

3. **Theo giai đoạn**:
   - Pre-COVID: 2.41%
   - COVID: 2.59% (tăng nhẹ)
   - Recovery: 2.02% (giảm)

4. **Youth Gap**:
   - Tất cả quốc gia: youth unemployment > overall unemployment
   - Gap trung bình: ~3.94 percentage points
   - Gap lớn nhất: South Korea (~5.27 pp)

5. **Correlation**:
   - GDP Growth vs Unemployment: r = -0.117 (yếu)
   - Inflation vs Unemployment: r = -0.259 (yếu-trung bình)
   - Pattern khác nhau giữa các quốc gia

6. **Reproducibility**:
   - Kết quả tính toán khớp với Agent 2 (nếu không có mismatches)
   - Dataset có thể tái tạo kết quả phân tích

7. **Machine Learning Preparation**:
   - Đã xác định potential features và target
   - Đã ghi chú các rủi ro data leakage
   - Cần đánh giá kỹ lưỡng feature selection khi xây model

### Bước tiếp theo:

- Xem reproducibility_check.md để biết chi tiết so sánh với Agent 2
- Dataset sẵn sàng cho Machine Learning (nếu cần)